# VLA robustness: cloud resource check and simulator smoke test
Upload this notebook to Google Colab. Select **Runtime → Change runtime type → GPU**. Run cells in order. No model weights or training datasets are downloaded. This is an unvalidated cloud setup, not a completed policy demo.

The first milestone is rendering an existing LIBERO task. Robot motion below uses dummy actions, not a learned policy. Save your results before the free runtime ends.


In [ ]:
import os, sys, json, shutil, subprocess
from pathlib import Path
if not Path('/content').exists() or 'google.colab' not in sys.modules:
    raise RuntimeError('Run this notebook in Colab, not on the laptop.')
ROOT = Path('/content/vla-robustness')
ROOT.mkdir(exist_ok=True)
OUT = ROOT / 'results'
OUT.mkdir(exist_ok=True)
def run(args, **kwargs):
    return subprocess.run(args, check=True, text=True, **kwargs)
probe = run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv'], capture_output=True)
print(probe.stdout)
free_gb = shutil.disk_usage(ROOT).free / 2**30
print('Python:', sys.version, '\nFree disk GiB:', round(free_gb, 1))
(OUT / 'hardware.json').write_text(json.dumps({'python':sys.version,'gpu':probe.stdout,'free_disk_gib':free_gb}, indent=2))
if free_gb < 10:
    raise RuntimeError('Need at least 10 GiB free for simulator setup; model setup needs more.')


## Install a separate Python 3.10 environment
LIBERO uses older robotics dependencies. An isolated environment avoids replacing Colab's notebook packages. Installation may take several minutes. No GPU model is loaded.


In [ ]:
run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'])
UV = shutil.which('uv')
run([UV, 'venv', '--python', '3.10', str(ROOT / 'venv')])
PYTHON = str(ROOT / 'venv/bin/python')
LIBERO = ROOT / 'LIBERO'
if not LIBERO.exists():
    run(['git', 'clone', '--depth', '1', 'https://github.com/Lifelong-Robot-Learning/LIBERO.git', str(LIBERO)])
revision = run(['git', '-C', str(LIBERO), 'rev-parse', 'HEAD'], capture_output=True).stdout.strip()
(OUT / 'libero-revision.txt').write_text(revision + '\n')
run([UV, 'pip', 'install', '--python', PYTHON,
     'numpy<2', 'torch', 'robosuite==1.4.1', 'bddl', 'easydict', 'cloudpickle',
     'gym==0.25.2', 'imageio[ffmpeg]', 'matplotlib', 'opencv-python-headless',
     'h5py', 'pyyaml', 'tqdm', 'einops'])
run([UV, 'pip', 'install', '--python', PYTHON, '--no-deps', '-e', str(LIBERO)])
frozen = run([UV, 'pip', 'freeze', '--python', PYTHON], capture_output=True).stdout
(OUT / 'requirements-resolved.txt').write_text(frozen)
# Preconfigure paths so LIBERO does not ask interactive setup questions.
import yaml
config_dir = ROOT / 'libero-config'
config_dir.mkdir(exist_ok=True)
base = LIBERO / 'libero' / 'libero'
(config_dir / 'config.yaml').write_text(yaml.safe_dump({
    'benchmark_root': str(base), 'bddl_files': str(base / 'bddl_files'),
    'init_states': str(base / 'init_files'), 'datasets': str(ROOT / 'datasets'),
    'assets': str(base / 'assets')}))


In [ ]:
smoke = r"""
import json
from pathlib import Path
import numpy as np
import imageio.v2 as imageio
from libero.libero import benchmark, get_libero_path
from libero.libero.envs import OffScreenRenderEnv
out = Path('results')
suite = benchmark.get_benchmark_dict()['libero_spatial']()
task = suite.get_task(0)
env = OffScreenRenderEnv(
    bddl_file_name=str(Path(get_libero_path('bddl_files')) / task.problem_folder / task.bddl_file),
    camera_heights=256, camera_widths=256)
try:
    env.seed(7)
    env.reset()
    obs = env.set_init_state(suite.get_task_init_states(0)[0])
    frames = []
    for step in range(30):
        obs, reward, done, info = env.step([0, 0, 0, 0, 0, 0, -1])
        frames.append(np.ascontiguousarray(obs['agentview_image'][::-1, ::-1]))
    imageio.mimsave(str(out / 'simulator-smoke.mp4'), frames, fps=10)
    (out / 'smoke.json').write_text(json.dumps({
        'task':task.language, 'seed':7, 'steps':30,
        'policy':'dummy_action', 'purpose':'rendering check only; not task evaluation'}, indent=2))
    print(task.language)
finally:
    env.close()
"""
(ROOT / 'smoke.py').write_text(smoke)
env = dict(os.environ, MUJOCO_GL='egl', PYOPENGL_PLATFORM='egl', LIBERO_CONFIG_PATH=str(config_dir))
result = subprocess.run(
    [PYTHON, '-u', 'smoke.py'], cwd=ROOT, env=env,
    text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
)
log_path = OUT / 'simulator-smoke.log'
log_path.write_text(result.stdout)
print(result.stdout)
print('Exit code:', result.returncode)
print('Saved log:', log_path)
if result.returncode:
    raise RuntimeError('Simulator failed. Share the printed log above, especially the final traceback.')


In [ ]:
from IPython.display import Video, display
display(Video(str(OUT / 'simulator-smoke.mp4'), embed=True))


## Save the artifacts
This downloads only the small smoke-test results. It does not download the virtual environment or simulator assets. If installation or rendering fails, keep the traceback: no policy benchmark has run yet.


In [ ]:
from google.colab import files
archive = shutil.make_archive('/content/vla-smoke-results', 'zip', OUT)
files.download(archive)


## Next gate: learned policy
Once rendering works, select a LIBERO-Spatial checkpoint and validate inference against the allocated GPU. Do not use the generic BridgeData checkpoint for this robot. Do not launch full-suite evaluation as a smoke test: upstream defaults to 50 trials per task.

A 16 GiB free GPU may need quantization; quantization changes the baseline and must be recorded. A 24 GiB GPU is a planning target, not a guarantee. Model inference, disturbances, and recovery are not implemented in this notebook yet.

Sources: [OpenVLA LIBERO setup](https://github.com/openvla/openvla#libero-simulation-benchmark-evaluations), [LIBERO](https://github.com/Lifelong-Robot-Learning/LIBERO).
